# CAPAR 2.0 — Google Colab Pipeline Simulation & Metadata Verification
## Simulasi FSM: 2-of-3 Window Persistence, EpisodeMeta Linking, & Disconnect Tau_Out Handler

Notebook ini mensimulasikan logika alur penuh sistem **CAPAR 2.0**:
1. **Candidate Onset**: Deteksi pertama kali skor anomali $S(t) \ge \tau_{in}$ (`DEVIATION_CANDIDATE`).
2. **2-of-3 Window Persistence**: Transisi ke `PERSISTENT_DEVIATION` bila minimal **2 dari 3 window terakhir** melampaui $\tau_{in}$.
3. **Disconnect / Device Dilepas Handler**: Apabila data terputus / gap $> 15$ menit sebelum recovery $\tau_{out}$, event ditutup paksa di window terakhir (`FORCE_CLOSED_TAU_OUT`) dengan `unresolved_reason` yang jelas.
4. **Koleksi EpisodeMeta**: Membuat & menghubungkan data meta peserta (tanggal, waktu, status, episode_id) dengan `EpisodeAnalysis`.

In [ ]:
import math
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

# Set matplotlib style for Google Colab
%matplotlib inline
plt.rcParams['figure.dpi'] = 120
print("✅ Core libraries loaded successfully.")

--- 
## 1. Konfigurasi Threshold & Simulasi FSM Engine

In [ ]:
TAU_IN = 1.86       # Threshold Entry Candidate Onset
TAU_OUT = 1.18      # Threshold Recovery Entry
TAU_NORMAL = 0.82   # Threshold Normal Baseline
DISCONNECT_TIMEOUT_MIN = 15 # Gap > 15 menit dianggap data terputus / device dilepas

class CAPARFSMSimulator:
    def __init__(self, tau_in=1.86, tau_out=1.18, tau_normal=0.82):
        self.tau_in = tau_in
        self.tau_out = tau_out
        self.tau_normal = tau_normal
        self.window_history = []  # Sliding window history 3-window
        self.episode_active = False
        self.current_state = 'BASELINE_COMPATIBLE'
        self.open_event = None
        self.last_ts = None
        self.events = []
        self.meta_collection = []
        self.analysis_collection = []

    def process_window(self, ts, score, participant_id="p-9669def075"):
        ts_ms = int(ts.timestamp() * 1000)
        
        # 1. Deteksi Data Terputus / Device Dilepas (> 15 menit gap saat episode aktif)
        if self.last_ts is not None and self.episode_active:
            gap_min = (ts - self.last_ts).total_seconds() / 60.0
            if gap_min > DISCONNECT_TIMEOUT_MIN:
                self.force_close_tau_out(ts_ms, reason="Data terputus / device dilepas sebelum titik tau_out (Force closed at last valid window)")
        
        self.last_ts = ts
        
        # Update sliding history 3-window (2-of-3 persistence check)
        self.window_history.append(score >= self.tau_in)
        if len(self.window_history) > 3:
            self.window_history.pop(0)
            
        count_in_last_3 = sum(self.window_history)

        # Evaluasi Transisi FSM
        if score >= self.tau_in:
            self.episode_active = True
            if count_in_last_3 >= 2:
                self.current_state = 'PERSISTENT_DEVIATION'
            else:
                self.current_state = 'DEVIATION_CANDIDATE'
            
            if not self.open_event:
                self.create_event(ts_ms, score, participant_id)
            else:
                self.update_event(ts_ms, score)
                
        elif self.episode_active:
            if count_in_last_3 >= 2 and score > self.tau_out:
                self.current_state = 'PERSISTENT_DEVIATION'
                self.update_event(ts_ms, score)
            elif score <= self.tau_normal:
                self.current_state = 'RECOVERED'
                self.close_event(ts_ms, score, status='closed')
                self.episode_active = False
                self.window_history = []
            elif score <= self.tau_out:
                self.current_state = 'RECOVERING'
                self.update_event(ts_ms, score)
            else:
                self.current_state = 'RECOVERING'
                self.update_event(ts_ms, score)
        else:
            self.current_state = 'BASELINE_COMPATIBLE'
            
        return self.current_state

    def create_event(self, ts_ms, score, participant_id):
        dt = datetime.fromtimestamp(ts_ms / 1000.0)
        event_id = f"ep_{ts_ms}"
        analysis_id = f"analysis_{ts_ms}"
        
        event_doc = {
            '_id': event_id,
            'participant_id': participant_id,
            'onset_time': ts_ms,
            'started_at': ts_ms,
            'candidate_at': ts_ms,
            'peak_time': ts_ms,
            'onset_score': score,
            'peak_score': score,
            'duration_ms': 300000, # 5 min window
            'status': 'open',
            'current_state': self.current_state,
            'window_count': 1,
            'resolved_time': None,
            'recovered_at': None,
            'unresolved_reason': None
        }
        self.open_event = event_doc
        self.events.append(event_doc)
        
        # Koleksi EpisodeMeta (Linked Meta Document)
        meta_doc = {
            'episode_id': event_id,
            'analysis_id': analysis_id,
            'participant_id': participant_id,
            'date': dt.strftime('%Y-%m-%d'),
            'time': dt.strftime('%H:%M:%S'),
            'onset_timestamp': ts_ms,
            'status': 'candidate' if self.current_state == 'DEVIATION_CANDIDATE' else 'persistent',
            'current_state': self.current_state,
            'peak_score': score,
            'duration_ms': 300000
        }
        self.meta_collection.append(meta_doc)

    def update_event(self, ts_ms, score):
        if not self.open_event: return
        self.open_event['window_count'] += 1
        if score > self.open_event['peak_score']:
            self.open_event['peak_score'] = score
            self.open_event['peak_time'] = ts_ms
        self.open_event['current_state'] = self.current_state
        self.open_event['duration_ms'] = ts_ms - self.open_event['onset_time'] + 300000
        
        for m in self.meta_collection:
            if m['episode_id'] == self.open_event['_id']:
                m['status'] = 'persistent' if 'PERSISTENT' in self.current_state else ('candidate' if 'CANDIDATE' in self.current_state else 'recovering')
                m['current_state'] = self.current_state
                m['peak_score'] = self.open_event['peak_score']
                m['duration_ms'] = self.open_event['duration_ms']

    def force_close_tau_out(self, ts_ms, reason):
        if not self.open_event: return
        last_valid_ts = self.open_event['onset_time'] + (self.open_event['window_count'] * 300000)
        
        self.open_event['status'] = 'closed'
        self.open_event['current_state'] = 'FORCE_CLOSED_TAU_OUT'
        self.open_event['recovery_entry_at'] = last_valid_ts
        self.open_event['recovered_at'] = last_valid_ts
        self.open_event['resolved_time'] = last_valid_ts
        self.open_event['unresolved_reason'] = reason
        self.open_event['duration_ms'] = last_valid_ts - self.open_event['onset_time']
        
        for m in self.meta_collection:
            if m['episode_id'] == self.open_event['_id']:
                m['status'] = 'recovered'
                m['current_state'] = 'FORCE_CLOSED_TAU_OUT'
                m['duration_ms'] = self.open_event['duration_ms']
        
        print(f"⚠️  [DISCONNECT HANDLER TRIGGERED]")
        print(f"    Episode ID      : {self.open_event['_id']}")
        print(f"    Reason          : {reason}")
        print(f"    Force Tau_Out At: {datetime.fromtimestamp(last_valid_ts/1000.0).strftime('%H:%M:%S')}")
        
        self.open_event = None
        self.episode_active = False

    def close_event(self, ts_ms, score, status='closed'):
        if not self.open_event: return
        self.open_event['status'] = status
        self.open_event['current_state'] = 'RECOVERED'
        self.open_event['recovered_at'] = ts_ms
        self.open_event['resolved_time'] = ts_ms
        self.open_event['duration_ms'] = ts_ms - self.open_event['onset_time']
        
        for m in self.meta_collection:
            if m['episode_id'] == self.open_event['_id']:
                m['status'] = 'recovered'
                m['current_state'] = 'RECOVERED'
                m['duration_ms'] = self.open_event['duration_ms']
                
        self.open_event = None

print("✅ CAPARFSMSimulator Class Initialized.")

--- 
## 2. Draf Simulasi Data Fisiologis (Menyertakan Gap Data / Device Terputus)

In [ ]:
base_time = datetime(2026, 8, 27, 16, 20, 0)
timestamps = []
scores = []

# 1. Baseline normal (5 window)
for i in range(5):
    timestamps.append(base_time + timedelta(minutes=5 * i))
    scores.append(0.5 + np.random.uniform(-0.1, 0.1))

# 2. Window 5: Onset Candidate (Score = 4.63 >= 1.86)
timestamps.append(base_time + timedelta(minutes=25))
scores.append(4.63)

# 3. Window 6: Score sedikit dip ke 1.50 (< tau_in tapi > tau_out)
timestamps.append(base_time + timedelta(minutes=30))
scores.append(1.50)

# 4. Window 7: Score high 3.85 (>= tau_in) -> 2 dari 3 window anomali => PERSISTENT_DEVIATION!
timestamps.append(base_time + timedelta(minutes=35))
scores.append(3.85)

# 5. Data Terputus / Device Dilepas (Simulasi Gap 30 Menit tanpa data recovery < tau_out)
timestamps.append(base_time + timedelta(minutes=65))
scores.append(1.40)

# 6. Recovery normal setelah terputus
timestamps.append(base_time + timedelta(minutes=70))
scores.append(0.90)
timestamps.append(base_time + timedelta(minutes=75))
scores.append(0.60)

df_sim = pd.DataFrame({'timestamp': timestamps, 'score': scores})

# Jalankan Simulasi FSM
sim = CAPARFSMSimulator(tau_in=TAU_IN, tau_out=TAU_OUT, tau_normal=TAU_NORMAL)
fsm_states = []

for idx, row in df_sim.iterrows():
    st = sim.process_window(row['timestamp'], row['score'])
    fsm_states.append(st)

df_sim['fsm_state'] = fsm_states
df_sim

--- 
## 3. Hasil Koleksi MongoDB: EpisodeMeta & AnomalyEvent Linked Documents

In [ ]:
print("=" * 80)
print("1. KOLEKSI EpisodeMeta (Metadata Ringkasan Peserta Terhubung)")
print("=" * 80)
print(json.dumps(sim.meta_collection, indent=2))

print("\n" + "=" * 80)
print("2. KOLEKSI AnomalyEvent (Detail Lifecycle Event & Disconnect Reason)")
print("=" * 80)
for ev in sim.events:
    print(json.dumps(ev, indent=2))

--- 
## 4. Visualisasi Grafik Matplotlib (Anomaly Score Trajectory & State Machine)

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 9), sharex=True, gridspec_kw={'height_ratios': [2.5, 1]})

time_labels = [t.strftime('%H:%M') for t in df_sim['timestamp']]
x_indices = np.arange(len(df_sim))

# Subplot 1: Score Trajectory & Threshold Lines
ax1.plot(x_indices, df_sim['score'], marker='o', color='#2b5c8f', linewidth=2.5, label='Anomaly Score S(t)')
ax1.axhline(y=TAU_IN, color='#d9534f', linestyle='--', linewidth=1.8, label=f'tau_in ({TAU_IN}) - Candidate Onset')
ax1.axhline(y=TAU_OUT, color='#f0ad4e', linestyle='-.', linewidth=1.8, label=f'tau_out ({TAU_OUT}) - Recovery Entry')
ax1.axhline(y=TAU_NORMAL, color='#5cb85c', linestyle=':', linewidth=1.8, label=f'tau_normal ({TAU_NORMAL}) - Baseline')

# Anotasi Titik Kunci
ax1.annotate('1. Candidate Onset\n(Score = 4.63 >= 1.86)', xy=(5, df_sim['score'].iloc[5]), xytext=(4.2, 5.2),
             arrowprops=dict(facecolor='#d9534f', shrink=0.08, width=1.5, headwidth=8),
             fontsize=9, fontweight='bold', color='#d9534f')

ax1.annotate('2. PERSISTENT_DEVIATION\n(2 dari 3 window >= tau_in)', xy=(7, df_sim['score'].iloc[7]), xytext=(6.0, 4.3),
             arrowprops=dict(facecolor='#8e44ad', shrink=0.08, width=1.5, headwidth=8),
             fontsize=9, fontweight='bold', color='#8e44ad')

ax1.annotate('3. DISCONNECT HANDLER\n(Gap > 15m -> Force Tau_Out)', xy=(8, df_sim['score'].iloc[8]), xytext=(7.0, 2.5),
             arrowprops=dict(facecolor='#c0392b', shrink=0.08, width=1.5, headwidth=8),
             fontsize=9, fontweight='bold', color='#c0392b')

ax1.set_title('CAPAR 2.0 Simulation: 2-of-3 Persistence, EpisodeMeta & Disconnect Tau_Out Handler', fontsize=13, fontweight='bold', pad=12)
ax1.set_ylabel('Anomaly Score Z', fontsize=11, fontweight='bold')
ax1.grid(True, linestyle='--', alpha=0.5)
ax1.legend(loc='upper right', frameon=True, facecolor='white', framealpha=0.9)

# Subplot 2: FSM State Transitions
state_mapping = {
    'BASELINE_COMPATIBLE': 0,
    'DEVIATION_CANDIDATE': 1,
    'PERSISTENT_DEVIATION': 2,
    'RECOVERING': 1.5,
    'FORCE_CLOSED_TAU_OUT': 0.5,
    'RECOVERED': 0
}
state_numeric = [state_mapping.get(s, 0) for s in df_sim['fsm_state']]

ax2.step(x_indices, state_numeric, where='post', color='#8e44ad', linewidth=2.5, label='FSM State')
ax2.set_yticks([0, 0.5, 1, 2])
ax2.set_yticklabels(['BASELINE / REC', 'TAU_OUT (DISCONNECT)', 'CANDIDATE', 'PERSISTENT'], fontsize=9, fontweight='bold')
ax2.set_xticks(x_indices)
ax2.set_xticklabels(time_labels, rotation=45, ha='right', fontsize=9)
ax2.set_xlabel('Waktu (HH:MM)', fontsize=11, fontweight='bold')
ax2.set_ylabel('CAPAR State', fontsize=11, fontweight='bold')
ax2.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()